# 16 Delta Table Time Travel

# Ver Versiones disponibles de la tabla

La siguiente celda ejecuta el notebook `15 Intro to Delta Tables` (`%run`) para reconstruir `people_delta` desde cero y dejarla disponible en este notebook.

In [0]:
%run "./15 Intro to Delta Tables"

> **Antes de ejecutar:** ¿Cuántas versiones dejó el notebook 15 en el historial?

In [0]:

DESCRIBE HISTORY people_delta;


# Consultas

In [ ]:
-- La versión 0 es la creación inicial de la tabla
SELECT * FROM people_delta VERSION AS OF 0;

Consultar la misma versión inicial de la tabla, ahora por timestamp (obtenido dinámicamente del historial en vez de un valor fijo):

In [ ]:
%python
hist = spark.sql("DESCRIBE HISTORY people_delta").orderBy("version")
ts_v0 = hist.filter("version = 0").select("timestamp").first()[0]
display(spark.sql(f"SELECT * FROM people_delta TIMESTAMP AS OF '{ts_v0}'"))

Reutilizamos el mismo timestamp de la versión 0 para filtrar además por `id`.

In [ ]:
%python
display(spark.sql(f"SELECT * FROM people_delta TIMESTAMP AS OF '{ts_v0}' WHERE id = 3"))

# Restaurar versiones

Primero podemos "cometer un error" y eliminar la tabla...

> **Antes de ejecutar:** Después de este DELETE sin WHERE, ¿cuántas filas quedarán? ¿Se pierden los datos para siempre?

In [0]:
DELETE  FROM people_delta

Luego revisamos la historia...

In [0]:
DESCRIBE HISTORY people_delta

Y restablecemos la tabla al estado inicial conocido (versión 0)...

In [ ]:
-- Restauramos al estado inicial conocido (versión 0 = creación de la tabla, antes de todos los UPDATE/DELETE)
RESTORE TABLE people_delta TO VERSION AS OF 0;

> **Antes de ejecutar:** Tras restaurar, ¿el RESTORE aparece como una nueva entrada en el historial o reescribe el pasado?

In [0]:
DESCRIBE HISTORY people_delta

Inspeccionamos los metadatos extendidos de la tabla.

In [0]:
DESCRIBE EXTENDED people_delta

Usamos OPTIMIZE para compactar archivos y ordenarlos físicamente por `age` (ZORDER).

> **Antes de ejecutar:** Con tan pocas filas, ¿esperas que ZORDER cambie el número de archivos? ¿Qué reportará la columna de métricas de la operación?

In [0]:
OPTIMIZE people_delta ZORDER BY (age);

Vemos los eventos

In [0]:
DESCRIBE HISTORY people_delta

Y podemos usar VACUUM para eliminar físicamente archivos de datos obsoletos que ya no pertenecen a ninguna versión activa.

> **Antes de ejecutar:** ¿Por qué crees que 168 horas (7 días) y no 0 horas? ¿Qué capacidad perderíamos si pudiéramos bajar la retención a 0?

En serverless, el período de retención por defecto no se puede reducir por debajo de este umbral de seguridad, precisamente para no romper el time travel de versiones recientes.

In [0]:
VACUUM people_delta RETAIN 168 HOURS;

Confirmamos que el `VACUUM` quedó registrado en el historial.

In [0]:
DESCRIBE HISTORY people_delta